In [ ]:
"""
OpenAlex OA Citation Extraction — Target-Aware TEI Version
==========================================================

Purpose
-------
Extract real citation contexts from OpenAlex cached GROBID/TEI XML for the
selected journals in JOURNALS for publication years 2023–2025.

TARGET-AWARE RULE
-----------------
The GROBID/TEI bibliographic reference occurrence itself is used to identify
citation targets via <ref type="bibr" target="..."> (and compatible
"biblio" values, defensively). For every resolvable citation occurrence in a
citation context, one target-aware instance is produced:

    the selected citation occurrence -> [TARGET_CITATION]
    every other citation occurrence  -> [OTHER_CITATION]

The original extracted context is preserved in `initial_sentence` in the
sentiment annotation CSV.

Two-phase workflow
------------------
FIRST RUN:
    1) retrieve eligible OpenAlex works
    2) reuse/download cached GROBID/TEI XML
    3) extract target-aware citation instances
    4) global deduplication
    5) deterministic approximately 50/50 split BEFORE polarity annotation

    SENTIMENT HALF:
      journal_citation_sentiment_annotation_candidates.csv
      - `sentence` contains [TARGET_CITATION]/[OTHER_CITATION]
      - `initial_sentence` contains the original extracted citation context
      - `polarity` is blank

    CITATION-FUNCTION HALF:
      journal_citation_function_real.csv
      - exact headers: source_dataset,citation,follow_up
      - `citation` contains [TARGET_CITATION]/[OTHER_CITATION]
      - `follow_up` is blank

SECOND RUN:
    After polarity is filled in the sentiment annotation CSV (0/1/2), rerun.
    No new OpenAlex requests are made. The script validates the completed
    annotation and creates:

      journal_citation_sentiment_real.csv

    with exact columns:
      Source_Paper_ID,sentence,is_citation,polarity,source,sampling_status

    `sentence` remains target-aware.

SAFETY
------
- Existing target-aware function/final sentiment files are never overwritten.
- Legacy/raw-text output files are detected and rejected rather than silently
  reused.
- A sentence is included only when every TEI bibliographic reference occurrence
  in it has exactly one usable `target` value. Missing/multiple TEI targets are
  logged and the whole sentence is excluded from strict target-aware output.
- Target-aware sibling instances derived from the same original sentence are
  kept in the SAME pre-annotation half.
"""

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 0. INSTALLATION (Colab)
# ============================================================
# If needed, run once in Colab:
# !pip install -q requests pandas lxml nltk tqdm

import os
import re
import time
import random
import hashlib
from copy import deepcopy
from pathlib import Path

import requests
import pandas as pd
from lxml import etree
from tqdm.auto import tqdm

import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)
except Exception:
    pass

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
random.seed(SEED)

JOURNALS = [
    "Scientometrics",                                                                # SCIM
    "Journal of Informetrics",                                                       # JOI
    "Quantitative Science Studies",                                                  # QSS
    "Journal of the Association for Information Science and Technology",             # JASIST
    "Frontiers in Research Metrics and Analytics",                                   # RMA
    "Iberoamerican Journal of Science Measurement and Communication",                # IJSMC"
    "Frontiers in Communication"                                                     # FCOMM"
]

PUBLICATION_YEARS = [2023, 2024, 2025]
FROM_PUBLICATION_DATE = "2023-01-01"
TO_PUBLICATION_DATE = "2025-12-31"

# Retained from the supplied script. Change only if you intentionally want a
# different corpus size.
ARTICLES_PER_JOURNAL_YEAR = 80

# Cap ORIGINAL citation sentences per source article before expanding each
# sentence into one target-aware row per TEI citation occurrence.
MAX_CITATION_SENTENCES_PER_ARTICLE = 50

MIN_SENTENCE_WORDS = 5
MAX_SENTENCE_WORDS = 120
MAX_SENTENCE_CHARS = 1500

TARGET_TOKEN = "[TARGET_CITATION]"
OTHER_TOKEN = "[OTHER_CITATION]"

# Strict mode preserves one universal meaning of TARGET_CITATION. A sentence is
# accepted only when each TEI bibliography ref in that sentence has exactly one
# usable target ID.
STRICT_TARGET_AWARE = True

BASE_DIR = Path(
    "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER"
)

OUTPUT_DIR = BASE_DIR / "journal_citation_openalex_2023_2025"
TEI_DIR = OUTPUT_DIR / "tei_xml"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEI_DIR.mkdir(parents=True, exist_ok=True)

ALL_METADATA_CSV = (
    OUTPUT_DIR / "openalex_all_candidate_articles_2023_2025.csv"
)

SELECTED_ARTICLES_CSV = (
    OUTPUT_DIR / "openalex_selected_articles_2023_2025.csv"
)

ANNOTATION_CSV = (
    OUTPUT_DIR / "journal_citation_sentiment_annotation_candidates.csv"
)

AUDIT_CSV = (
    OUTPUT_DIR / "openalex_citation_extraction_audit.csv"
)

TARGET_AUDIT_CSV = (
    OUTPUT_DIR / "openalex_target_aware_sentence_audit.csv"
)

FINAL_MERGE_READY_CSV = (
    BASE_DIR / "journal_citation_sentiment_real.csv"
)

FINAL_CITATION_FUNCTION_CSV = (
    BASE_DIR / "journal_citation_function_real.csv"
)

CITATION_FUNCTION_COLUMNS = [
    "source_dataset",
    "citation",
    "follow_up",
]

# Exact final sentiment-training schema.
FINAL_COLUMNS = [
    "Source_Paper_ID",
    "sentence",
    "is_citation",
    "polarity",
    "source",
    "sampling_status",
]

# Annotation/audit-only columns. `initial_sentence` is deliberately retained
# only in the annotation CSV, not in the six-column final sentiment export.
ANNOTATION_EXTRA_COLUMNS = [
    "initial_sentence",
    "journal",
    "doi",
    "title",
    "publication_year",
    "publication_date",
    "openalex_work_id",
    "n_citations_in_sentence",
    "is_open_access",
    "license",
    "oa_status",
]

ANNOTATION_COLUMNS = [
    "Source_Paper_ID",
    "sentence",
    "initial_sentence",
    "is_citation",
    "polarity",
    "source",
    "sampling_status",
    "journal",
    "doi",
    "title",
    "publication_year",
    "publication_date",
    "openalex_work_id",
    "n_citations_in_sentence",
    "is_open_access",
    "license",
    "oa_status",
]

OPENALEX_API_BASE = "https://api.openalex.org"
OPENALEX_CONTENT_BASE = "https://content.openalex.org"

REQUEST_TIMEOUT = 90
MAX_RETRIES = 6
PER_PAGE = 100

In [ ]:
# ============================================================
# 2. GENERAL / WORKFLOW HELPERS
# ============================================================


def normalize_whitespace(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def normalized_sentence_key(text):
    return normalize_whitespace(text).casefold()


def stable_seed(*parts):
    payload = "||".join(map(str, parts))
    digest = hashlib.sha256(payload.encode("utf-8")).hexdigest()
    return (SEED + int(digest[:8], 16)) % (2**32 - 1)


def _workflow_stable_seed(*parts):
    return stable_seed(*parts)


def _count_exact_token(text, token):
    return str(text).count(token)


def _is_target_aware_series(series):
    if series is None or len(series) == 0:
        return False
    values = series.fillna("").astype(str)
    return bool(values.map(lambda x: _count_exact_token(x, TARGET_TOKEN) == 1).all())


def _assert_target_aware_sentences(series, label="sentence"):
    values = series.fillna("").astype(str)
    bad = values.map(lambda x: _count_exact_token(x, TARGET_TOKEN) != 1)
    if bad.any():
        raise ValueError(
            f"Υπάρχουν {int(bad.sum())} {label} χωρίς ακριβώς ένα {TARGET_TOKEN}."
        )


def remove_redundant_single_citation_wrappers(text: str) -> str:
    """
    Remove wrappers only when they contain exactly one special citation token.

    Examples:
        ([TARGET_CITATION]) -> [TARGET_CITATION]
        ([OTHER_CITATION])  -> [OTHER_CITATION]
        [[TARGET_CITATION]] -> [TARGET_CITATION]

    Clusters remain intact:
        ([TARGET_CITATION]; [OTHER_CITATION])
    """
    for token in (TARGET_TOKEN, OTHER_TOKEN):
        escaped = re.escape(token)
        text = re.sub(rf"\(\s*{escaped}\s*\)", token, text)
        text = re.sub(rf"\[\s*{escaped}\s*\]", token, text)
    return text


def clean_rendered_sentence(text):
    text = normalize_whitespace(text)
    # Light punctuation cleanup only.
    text = re.sub(r"\s+([,.;:!?%])", r"\1", text)
    return text.strip()


def _annotation_file_state(csv_path):
    """Return missing, legacy_incompatible, unlabeled, partial, complete, invalid."""
    csv_path = Path(csv_path)

    if not csv_path.exists():
        return "missing", None

    df = pd.read_csv(csv_path)

    # Old/raw annotation files must not silently enter the new target-aware flow.
    if "initial_sentence" not in df.columns or "sentence" not in df.columns:
        return "legacy_incompatible", df

    if not _is_target_aware_series(df["sentence"]):
        return "legacy_incompatible", df

    if "polarity" not in df.columns or df.empty:
        return "invalid", df

    raw = df["polarity"]
    raw_nonblank = raw.notna() & raw.astype(str).str.strip().ne("")
    n_raw_filled = int(raw_nonblank.sum())

    if n_raw_filled == 0:
        return "unlabeled", df

    numeric = pd.to_numeric(raw, errors="coerce")
    n_numeric_filled = int(numeric.notna().sum())

    if n_numeric_filled != n_raw_filled:
        return "invalid", df

    if n_numeric_filled < len(df):
        return "partial", df

    if not (numeric == numeric.astype(int)).all():
        return "invalid", df

    invalid_values = set(numeric.astype(int).unique()) - {0, 1, 2}
    if invalid_values:
        return "invalid", df

    return "complete", df


def _validate_sentiment_annotation(annotated):
    required = set(ANNOTATION_COLUMNS)
    missing = required - set(annotated.columns)
    if missing:
        raise ValueError(f"Λείπουν στήλες: {sorted(missing)}")

    annotated = annotated.copy()

    _assert_target_aware_sentences(annotated["sentence"], "annotation sentences")

    if annotated["initial_sentence"].isna().any():
        raise ValueError("Υπάρχουν κενές initial_sentence.")

    annotated["initial_sentence"] = (
        annotated["initial_sentence"].astype(str).str.strip()
    )
    if annotated["initial_sentence"].eq("").any():
        raise ValueError("Υπάρχουν κενές initial_sentence.")

    raw = annotated["polarity"]
    if raw.isna().any() or raw.astype(str).str.strip().eq("").any():
        raise ValueError("Υπάρχουν sentiment citations χωρίς polarity.")

    numeric = pd.to_numeric(raw, errors="raise")
    if not (numeric == numeric.astype(int)).all():
        raise ValueError("Η polarity πρέπει να περιέχει μόνο ακέραιες τιμές 0, 1, 2.")

    annotated["polarity"] = numeric.astype(int)

    invalid = set(annotated["polarity"].unique()) - {0, 1, 2}
    if invalid:
        raise ValueError(f"Μη έγκυρες polarity τιμές: {sorted(invalid)}")

    annotated["is_citation"] = pd.to_numeric(
        annotated["is_citation"], errors="raise"
    ).astype(int)

    if not (annotated["is_citation"] == 1).all():
        raise ValueError("Βρέθηκε journal row με is_citation != 1.")

    if annotated["sentence"].isna().any():
        raise ValueError("Υπάρχουν κενές target-aware citation sentences.")

    annotated["sentence"] = annotated["sentence"].astype(str).str.strip()
    if annotated["sentence"].eq("").any():
        raise ValueError("Υπάρχουν κενές target-aware citation sentences.")

    annotated["sampling_status"] = (
        "external_real_journal_annotated_openalex_tei_target_aware"
    )

    # Duplicate/conflict check on target-aware model input.
    annotated["_norm"] = annotated["sentence"].map(normalized_sentence_key)
    conflict_counts = annotated.groupby("_norm")["polarity"].nunique()
    conflicting_norms = set(conflict_counts[conflict_counts > 1].index)
    if conflicting_norms:
        raise ValueError(
            "Βρέθηκαν ίδιες target-aware normalized sentences με διαφορετικό "
            f"polarity: {len(conflicting_norms)}. Έλεγξέ τες πριν το export."
        )

    annotated = (
        annotated
        .drop_duplicates(subset=["_norm"], keep="first")
        .drop(columns=["_norm"])
        .reset_index(drop=True)
    )

    if annotated[FINAL_COLUMNS].isna().any().any():
        raise ValueError("Υπάρχουν κενές τιμές στις απαιτούμενες sentiment στήλες.")

    return annotated


def _validate_existing_target_aware_function_file(path):
    existing = pd.read_csv(path)
    missing = set(CITATION_FUNCTION_COLUMNS) - set(existing.columns)
    if missing:
        raise ValueError(
            "Υπάρχει ήδη journal_citation_function_real.csv αλλά λείπουν "
            f"στήλες: {sorted(missing)}. Δεν γίνεται overwrite."
        )
    if not _is_target_aware_series(existing["citation"]):
        raise RuntimeError(
            "Υπάρχει ήδη journal_citation_function_real.csv από παλαιότερο/raw "
            "workflow και η citation δεν είναι target-aware. Μετονόμασε ή "
            "μετακίνησέ το πριν τρέξεις το νέο target-aware extraction."
        )
    return existing[CITATION_FUNCTION_COLUMNS].copy()


def _validate_existing_target_aware_final_sentiment(path):
    existing = pd.read_csv(path)
    missing = set(FINAL_COLUMNS) - set(existing.columns)
    if missing:
        raise ValueError(
            "Υπάρχει ήδη journal_citation_sentiment_real.csv αλλά λείπουν "
            f"στήλες: {sorted(missing)}. Δεν γίνεται overwrite."
        )
    if not _is_target_aware_series(existing["sentence"]):
        raise RuntimeError(
            "Υπάρχει ήδη journal_citation_sentiment_real.csv από παλαιότερο/raw "
            "workflow και η sentence δεν είναι target-aware. Μετονόμασε ή "
            "μετακίνησέ το πριν δημιουργήσεις το νέο target-aware final file."
        )

    existing_polarity = pd.to_numeric(existing["polarity"], errors="coerce")
    if existing_polarity.isna().any() or not existing_polarity.isin([0, 2]).all():
        raise RuntimeError(
            "Υπάρχει ήδη journal_citation_sentiment_real.csv που περιέχει "
            "Neutral ή μη έγκυρες polarity τιμές. Το νέο workflow επιτρέπει "
            "στο τελικό sentiment CSV μόνο 0=Negative και 2=Positive. "
            "Μετονόμασε ή μετακίνησε το υπάρχον αρχείο και ξανατρέξε."
        )

    return existing[FINAL_COLUMNS].copy()


def _export_sentiment_final(annotated):
    # Validate the COMPLETE annotation file first. The candidates file may and
    # should contain all three labels: 0=Negative, 1=Neutral, 2=Positive.
    annotated = _validate_sentiment_annotation(annotated)

    # FINAL sentiment training file: retain ONLY Negative and Positive.
    # Neutral rows remain in the annotation/audit CSV but are intentionally
    # excluded from journal_citation_sentiment_real.csv.
    sentiment_df = (
        annotated.loc[annotated["polarity"].isin([0, 2]), FINAL_COLUMNS]
        .copy()
        .reset_index(drop=True)
    )

    if FINAL_MERGE_READY_CSV.exists():
        existing = _validate_existing_target_aware_final_sentiment(
            FINAL_MERGE_READY_CSV
        )
        print("\nFINAL TARGET-AWARE SENTIMENT DATASET ALREADY EXISTS — PRESERVED")
        print(FINAL_MERGE_READY_CSV)
        print("Δεν έγινε overwrite.")
        print("Existing rows:", len(existing))
        return existing

    sentiment_df.to_csv(
        FINAL_MERGE_READY_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    n_neutral_excluded = int((annotated["polarity"] == 1).sum())

    print("\nSENTIMENT ANNOTATION COMPLETE — TARGET-AWARE FINAL EXPORT")
    print(FINAL_MERGE_READY_CSV)
    print("Rows retained (Negative + Positive only):", len(sentiment_df))
    print("Neutral rows excluded from final sentiment CSV:", n_neutral_excluded)
    print("Polarity distribution:")
    print(sentiment_df["polarity"].value_counts().sort_index())

    return sentiment_df


def _split_before_annotation_50_50(candidates):
    """
    Deterministic approximately 50/50 split BEFORE polarity annotation.

    IMPORTANT: all target-aware sibling instances derived from the SAME
    original citation context are kept in the SAME half. This prevents the same
    original sentence from appearing once in the sentiment half and once in the
    function half with only TARGET_CITATION moved to a different position.
    """
    required = {
        "Source_Paper_ID",
        "sentence",
        "initial_sentence",
        "journal",
        "publication_year",
        "title",
    }
    missing = required - set(candidates.columns)
    if missing:
        raise ValueError(f"Λείπουν στήλες για το split: {sorted(missing)}")

    candidates = candidates.copy().reset_index(drop=True)
    candidates["_initial_norm"] = candidates["initial_sentence"].map(
        normalized_sentence_key
    )
    candidates["_context_group"] = (
        candidates["Source_Paper_ID"].astype(str)
        + "||"
        + candidates["_initial_norm"]
    )

    sentiment_parts = []
    function_parts = []

    grouped = candidates.groupby(
        ["journal", "publication_year"],
        sort=True,
        dropna=False,
    )

    for (journal, year), stratum in grouped:
        group_ids = stratum["_context_group"].drop_duplicates().tolist()
        rng = random.Random(
            _workflow_stable_seed(journal, year, "target_aware_group_split")
        )
        rng.shuffle(group_ids)

        sent_ids = []
        func_ids = []
        n_sent = 0
        n_func = 0

        group_sizes = stratum.groupby("_context_group").size().to_dict()

        for gid in group_ids:
            size = int(group_sizes[gid])
            # Greedy row-count balancing while preserving whole context groups.
            if n_sent <= n_func:
                sent_ids.append(gid)
                n_sent += size
            else:
                func_ids.append(gid)
                n_func += size

        sentiment_parts.append(
            stratum[stratum["_context_group"].isin(sent_ids)].copy()
        )
        function_parts.append(
            stratum[stratum["_context_group"].isin(func_ids)].copy()
        )

    empty = candidates.iloc[0:0].copy()
    sentiment_half = (
        pd.concat(sentiment_parts, ignore_index=True)
        if sentiment_parts else empty.copy()
    )
    function_half = (
        pd.concat(function_parts, ignore_index=True)
        if function_parts else empty.copy()
    )

    # Safety: no source-context group may cross halves.
    sent_groups = set(sentiment_half["_context_group"])
    func_groups = set(function_half["_context_group"])
    overlap = sent_groups & func_groups
    if overlap:
        raise RuntimeError(
            f"Βρέθηκαν {len(overlap)} original contexts και στα δύο halves."
        )

    if len(sentiment_half) + len(function_half) != len(candidates):
        raise RuntimeError("Το target-aware split έχασε ή διπλασίασε εγγραφές.")

    sentiment_half = sentiment_half.sample(
        frac=1,
        random_state=SEED,
    ).reset_index(drop=True)
    function_half = function_half.sample(
        frac=1,
        random_state=SEED + 1,
    ).reset_index(drop=True)

    helper_cols = ["_initial_norm", "_context_group"]
    sentiment_half = sentiment_half.drop(columns=helper_cols)
    function_half = function_half.drop(columns=helper_cols)

    return sentiment_half, function_half


def _create_function_dataset(function_half):
    function_half = function_half.copy()
    _assert_target_aware_sentences(
        function_half["sentence"], "function target-aware sentences"
    )

    title_clean = function_half["title"].fillna("").astype(str).str.strip()
    journal_clean = function_half["journal"].fillna("").astype(str).str.strip()
    source_dataset_values = title_clean.mask(title_clean.eq(""), journal_clean)

    if source_dataset_values.eq("").any():
        n_missing = int(source_dataset_values.eq("").sum())
        raise ValueError(
            f"Υπάρχουν {n_missing} function rows χωρίς title και χωρίς journal."
        )

    function_df = pd.DataFrame({
        "source_dataset": source_dataset_values,
        "citation": function_half["sentence"].astype(str).str.strip(),
        "follow_up": pd.NA,
    })

    if function_df["citation"].eq("").any():
        raise ValueError("Υπάρχουν κενές citation τιμές στο function dataset.")

    return function_df[CITATION_FUNCTION_COLUMNS].copy()


def _save_function_dataset_once(function_df):
    if FINAL_CITATION_FUNCTION_CSV.exists():
        existing = _validate_existing_target_aware_function_file(
            FINAL_CITATION_FUNCTION_CSV
        )
        print("\nPROTECTION: target-aware journal_citation_function_real.csv υπάρχει ήδη.")
        print("ΔΕΝ αντικαθίσταται και ΔΕΝ αλλάζει η follow_up.")
        print("Existing rows:", len(existing))
        print("Saved file preserved:", FINAL_CITATION_FUNCTION_CSV)
        return existing, False

    function_df = function_df[CITATION_FUNCTION_COLUMNS].copy()
    function_df.to_csv(
        FINAL_CITATION_FUNCTION_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print("\nCreated TARGET-AWARE citation-function dataset:")
    print(FINAL_CITATION_FUNCTION_CSV)
    return function_df, True

In [ ]:
# ============================================================
# 3. TWO-PHASE WORKFLOW CONTROLLER
# ============================================================

_annotation_state, _existing_annotation_df = _annotation_file_state(ANNOTATION_CSV)

if _annotation_state == "legacy_incompatible":
    raise RuntimeError(
        "Βρέθηκε παλαιότερο journal_citation_sentiment_annotation_candidates.csv "
        "που δεν είναι target-aware ή δεν έχει initial_sentence. Μετονόμασε/"
        "μετακίνησέ το πριν τρέξεις το νέο script. Η παλιά annotation δεν μπορεί "
        "να επαναχρησιμοποιηθεί με ασφάλεια επειδή τώρα δημιουργείται μία row ανά "
        "TEI citation target occurrence."
    )

if _annotation_state == "complete":
    print("\nΒρέθηκε πλήρως συμπληρωμένο TARGET-AWARE sentiment annotation CSV.")
    print("Παραλείπονται όλα τα OpenAlex requests.")
    _export_sentiment_final(_existing_annotation_df)
    raise SystemExit(0)

if _annotation_state == "partial":
    raise ValueError(
        "Το target-aware journal_citation_sentiment_annotation_candidates.csv "
        "είναι ΜΕΡΙΚΩΣ annotated. Συμπλήρωσε όλα τα polarity και ξανατρέξε."
    )

if _annotation_state == "invalid":
    raise ValueError(
        "Το υπάρχον target-aware annotation CSV έχει μη έγκυρη δομή ή polarity."
    )

if _annotation_state == "unlabeled":
    # At this point the first extraction/split has already happened. Do not
    # re-extract or re-split.
    if not FINAL_CITATION_FUNCTION_CSV.exists():
        raise RuntimeError(
            "Υπάρχει unlabeled target-aware annotation CSV αλλά λείπει το "
            "journal_citation_function_real.csv. Για λόγους αναπαραγωγιμότητας "
            "δεν γίνεται νέο split. Επανέφερε το function file ή ξεκίνα σκόπιμα "
            "νέο extraction αφού μετακινήσεις το annotation CSV."
        )

    _validate_existing_target_aware_function_file(FINAL_CITATION_FUNCTION_CSV)
    print("\nΥπάρχει ήδη το TARGET-AWARE sentiment half με κενό polarity.")
    print("Το target-aware function half υπάρχει επίσης και προστατεύεται.")
    print("Δεν γίνονται OpenAlex requests και δεν αντικαθίσταται κανένα αρχείο.")
    print("Συμπλήρωσε polarity στο:")
    print(ANNOTATION_CSV)
    raise SystemExit(0)

# If annotation is missing, existing derived files make a fresh split unsafe.
if _annotation_state == "missing":
    if FINAL_CITATION_FUNCTION_CSV.exists():
        # Validate only to provide the right error message (target-aware vs legacy).
        _validate_existing_target_aware_function_file(FINAL_CITATION_FUNCTION_CSV)
        raise RuntimeError(
            "Λείπει το target-aware annotation CSV αλλά υπάρχει ήδη target-aware "
            "journal_citation_function_real.csv. Επανέφερε το annotation CSV ή "
            "μετονόμασε/μετακίνησε το function file αν θέλεις σκόπιμα νέο split."
        )

    if FINAL_MERGE_READY_CSV.exists():
        _validate_existing_target_aware_final_sentiment(FINAL_MERGE_READY_CSV)
        raise RuntimeError(
            "Λείπει το annotation CSV αλλά υπάρχει ήδη target-aware "
            "journal_citation_sentiment_real.csv. Μετονόμασε/μετακίνησε το final "
            "file μόνο αν θέλεις σκόπιμα να ξεκινήσεις νέο corpus."
        )

print("\nΔεν υπάρχει target-aware sentiment annotation CSV — ξεκινά extraction.")

In [ ]:
# ============================================================
# 4. OPENALEX API KEY
# ============================================================

from google.colab import userdata

OPENALEX_API_KEY = userdata.get("OPENALEX_API_KEY")

if not OPENALEX_API_KEY:
    raise ValueError("Δεν βρέθηκε το OPENALEX_API_KEY στα Colab Secrets.")

In [ ]:
# ============================================================
# 5. HTTP HELPERS WITH RETRIES
# ============================================================

session = requests.Session()
session.headers.update({
    "User-Agent": "PhD-citation-research/1.0"
})


def request_with_retries(url, params=None, stream=False):
    last_error = None

    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT,
                stream=stream,
            )

            if response.status_code == 200:
                return response

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                wait = (
                    int(retry_after)
                    if retry_after and retry_after.isdigit()
                    else min(2 ** attempt, 60)
                )
                print(f"OpenAlex 429. Retry σε {wait}s...")
                time.sleep(wait)
                continue

            if 500 <= response.status_code < 600:
                wait = min(2 ** attempt, 60)
                time.sleep(wait)
                continue

            response.raise_for_status()

        except requests.RequestException as exc:
            last_error = exc
            wait = min(2 ** attempt, 60)
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(wait)

    if last_error is not None:
        raise last_error

    raise RuntimeError(f"Αποτυχία request: {url}")


def openalex_json(endpoint, params=None):
    params = dict(params or {})
    params["api_key"] = OPENALEX_API_KEY

    response = request_with_retries(
        f"{OPENALEX_API_BASE}/{endpoint.lstrip('/')}",
        params=params,
    )
    return response.json()

In [ ]:
# ============================================================
# 6. RESOLVE JOURNALS TO EXACT OPENALEX SOURCE IDs
# ============================================================


def resolve_source_exact(journal_name):
    data = openalex_json(
        "sources",
        params={
            "search": journal_name,
            "filter": "type:journal",
            "per_page": 10,
            "select": "id,display_name,issn_l,issn,type,works_count",
        },
    )

    results = data.get("results", [])

    exact = [
        r for r in results
        if str(r.get("display_name", "")).strip().casefold()
        == journal_name.strip().casefold()
    ]

    if not exact:
        candidates = [
            (r.get("display_name"), r.get("id"), r.get("issn_l"))
            for r in results
        ]
        raise ValueError(
            f"Δεν βρέθηκε exact OpenAlex source για '{journal_name}'.\n"
            f"Top candidates: {candidates}"
        )

    exact = sorted(
        exact,
        key=lambda x: x.get("works_count") or 0,
        reverse=True,
    )

    source = exact[0]
    source_id = source["id"].rstrip("/").split("/")[-1]

    print("\nResolved journal:")
    print("  Name:", source.get("display_name"))
    print("  OpenAlex Source ID:", source_id)
    print("  ISSN-L:", source.get("issn_l"))
    print("  ISSNs:", source.get("issn"))

    return {
        "journal": journal_name,
        "source_id": source_id,
        "issn_l": source.get("issn_l"),
        "issn": source.get("issn"),
    }


journal_sources = {
    journal: resolve_source_exact(journal)
    for journal in JOURNALS
}

In [ ]:
# ============================================================
# 7. FETCH ALL ELIGIBLE 2023-2025 ARTICLES
# ============================================================


def fetch_eligible_works(journal_name, source_id):
    filter_string = ",".join([
        f"primary_location.source.id:{source_id}",
        f"from_publication_date:{FROM_PUBLICATION_DATE}",
        f"to_publication_date:{TO_PUBLICATION_DATE}",
        "type:article",
        "language:en",
        "open_access.is_oa:true",
        "is_retracted:false",
        "is_paratext:false",
        "has_content.grobid_xml:true",
    ])

    cursor = "*"
    rows = []

    while cursor:
        data = openalex_json(
            "works",
            params={
                "filter": filter_string,
                "per_page": PER_PAGE,
                "cursor": cursor,
                "select": (
                    "id,doi,title,publication_year,publication_date,"
                    "type,language,is_retracted,primary_location,"
                    "best_oa_location,open_access,referenced_works_count"
                ),
            },
        )

        results = data.get("results", [])

        for work in results:
            primary_location = work.get("primary_location") or {}
            best_oa_location = work.get("best_oa_location") or {}
            open_access = work.get("open_access") or {}

            rows.append({
                "journal": journal_name,
                "openalex_work_id": (
                    work.get("id", "").rstrip("/").split("/")[-1]
                ),
                "doi": work.get("doi"),
                "title": work.get("title"),
                "publication_year": work.get("publication_year"),
                "publication_date": work.get("publication_date"),
                "type": work.get("type"),
                "language": work.get("language"),
                "is_retracted": work.get("is_retracted"),
                "referenced_works_count": work.get("referenced_works_count"),
                "is_open_access": open_access.get("is_oa"),
                "license": (
                    best_oa_location.get("license")
                    or primary_location.get("license")
                ),
                "oa_status": open_access.get("oa_status"),
                "landing_page_url": primary_location.get("landing_page_url"),
            })

        next_cursor = (data.get("meta") or {}).get("next_cursor")
        if not results or not next_cursor:
            break
        cursor = next_cursor

    works_df = pd.DataFrame(rows)

    if works_df.empty:
        print(
            f"\n{journal_name}: 0 eligible works with GROBID XML in 2023-2025."
        )
        return works_df

    works_df = works_df[
        works_df["publication_year"].isin(PUBLICATION_YEARS)
    ].copy()

    if not works_df.empty and not works_df["is_open_access"].fillna(False).all():
        raise ValueError(
            f"Βρέθηκε non-OA work στο {journal_name}, παρότι έχει εφαρμοστεί "
            "open_access.is_oa:true."
        )

    works_df = works_df.drop_duplicates(
        subset=["openalex_work_id"],
        keep="first",
    ).reset_index(drop=True)

    print(f"\n{journal_name}")
    print("Eligible articles:", len(works_df))
    print("By year:")
    print(works_df["publication_year"].value_counts().sort_index())

    return works_df


all_work_frames = []

for journal in JOURNALS:
    source_id = journal_sources[journal]["source_id"]
    journal_df = fetch_eligible_works(journal, source_id)
    if not journal_df.empty:
        all_work_frames.append(journal_df)

if not all_work_frames:
    raise RuntimeError("Δεν βρέθηκαν eligible articles στα επιλεγμένα journals.")

all_metadata = pd.concat(all_work_frames, ignore_index=True)
assert all_metadata["publication_year"].isin([2023, 2024, 2025]).all()

all_metadata.to_csv(
    ALL_METADATA_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("\nSaved all eligible metadata:")
print(ALL_METADATA_CSV)

In [ ]:
# ============================================================
# 8. BALANCED, REPRODUCIBLE ARTICLE SAMPLE
# ============================================================

selected_groups = []

for (journal, year), group in all_metadata.groupby(
    ["journal", "publication_year"],
    sort=True,
):
    group = group.sort_values(
        ["publication_date", "openalex_work_id"]
    ).reset_index(drop=True)

    n_take = min(ARTICLES_PER_JOURNAL_YEAR, len(group))

    sampled = group.sample(
        n=n_take,
        random_state=stable_seed(journal, year),
        replace=False,
    )

    selected_groups.append(sampled)

selected_articles = (
    pd.concat(selected_groups, ignore_index=True)
    .sort_values(["journal", "publication_year", "openalex_work_id"])
    .reset_index(drop=True)
)

selected_articles.to_csv(
    SELECTED_ARTICLES_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("\nSELECTED ARTICLES")
print(selected_articles.groupby(["journal", "publication_year"]).size())
print("\nTotal selected articles:", len(selected_articles))
print("Saved:", SELECTED_ARTICLES_CSV)

In [ ]:
# ============================================================
# 9. DOWNLOAD CACHED OPENALEX GROBID/TEI XML
# ============================================================


def safe_folder_name(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("_")


def download_grobid_xml(work_row):
    work_id = work_row["openalex_work_id"]
    journal = work_row["journal"]
    year = int(work_row["publication_year"])

    journal_dir = TEI_DIR / safe_folder_name(journal) / str(year)
    journal_dir.mkdir(parents=True, exist_ok=True)

    xml_path = journal_dir / f"{work_id}.grobid.xml"

    if xml_path.exists() and xml_path.stat().st_size > 0:
        return xml_path, "cached_local"

    url = f"{OPENALEX_CONTENT_BASE}/works/{work_id}.grobid-xml"

    response = request_with_retries(
        url,
        params={"api_key": OPENALEX_API_KEY},
    )

    xml_bytes = response.content
    if not xml_bytes.strip():
        raise ValueError("Empty GROBID XML response.")

    xml_path.write_bytes(xml_bytes)
    return xml_path, "downloaded"

In [ ]:
# ============================================================
# 10. TARGET-AWARE EXTRACTION DIRECTLY FROM TEI <ref ... target>
# ============================================================

TEI_NS = {
    "tei": "http://www.tei-c.org/ns/1.0"
}

# GROBID/OpenAlex files in the supplied pipeline use type='bibr'. We also accept
# 'biblio' defensively because that wording appears in GROBID TEI documentation.
BIB_REF_XPATH = (
    ".//tei:ref[@type='bibr' or @type='biblio']"
)

PLACEHOLDER_RE = re.compile(r"__OA_BIBREF_(\d{5})__")


def normalize_tei_target_id(value):
    value = normalize_whitespace(value)
    if not value:
        return ""

    # Common TEI forms: #b12, b12, URL#b12.
    if "#" in value:
        value = value.rsplit("#", 1)[-1]
    return value.strip()


def parse_tei_targets(raw_target):
    """
    Parse the TEI target attribute conservatively.

    Whitespace-separated targets are treated as multiple bibliography targets.
    Duplicate IDs are removed while preserving order.
    """
    raw_target = normalize_whitespace(raw_target or "")
    if not raw_target:
        return []

    parts = re.split(r"\s+", raw_target)
    targets = []
    seen = set()
    for part in parts:
        tid = normalize_tei_target_id(part)
        if tid and tid not in seen:
            seen.add(tid)
            targets.append(tid)
    return targets


def paragraph_with_target_placeholders(paragraph):
    """
    Replace every bibliographic <ref> with a punctuation-free unique placeholder
    BEFORE sentence tokenization, while retaining its rendered surface and TEI
    target metadata in a side dictionary.
    """
    p = deepcopy(paragraph)

    bibr_refs = p.xpath(BIB_REF_XPATH, namespaces=TEI_NS)
    if not bibr_refs:
        return None, None

    ref_meta = {}

    for ref_index, ref in enumerate(bibr_refs, start=1):
        citation_surface = normalize_whitespace("".join(ref.itertext()))
        if not citation_surface:
            citation_surface = "[CITATION]"

        raw_target = ref.get("target", "")
        target_ids = parse_tei_targets(raw_target)

        placeholder = f"__OA_BIBREF_{ref_index:05d}__"

        ref_meta[ref_index] = {
            "ref_index": ref_index,
            "placeholder": placeholder,
            "surface": citation_surface,
            "raw_target": raw_target,
            "target_ids": target_ids,
        }

        original_tail = ref.tail
        for child in list(ref):
            ref.remove(child)

        ref.text = f" {placeholder} "
        ref.tail = original_tail

    marked_paragraph = normalize_whitespace("".join(p.itertext()))
    return marked_paragraph, ref_meta


def refs_in_marked_sentence(marked_sentence, ref_meta):
    refs = []
    for m in PLACEHOLDER_RE.finditer(marked_sentence):
        idx = int(m.group(1))
        meta = ref_meta.get(idx)
        if meta is None:
            return None
        refs.append(meta)
    return refs


def render_initial_sentence(marked_sentence, refs):
    text = marked_sentence
    for ref in refs:
        text = text.replace(ref["placeholder"], ref["surface"])
    return clean_rendered_sentence(text)


def render_target_aware_sentence(marked_sentence, refs, target_occurrence_index):
    text = marked_sentence

    for occurrence_index, ref in enumerate(refs):
        token = (
            TARGET_TOKEN
            if occurrence_index == target_occurrence_index
            else OTHER_TOKEN
        )
        text = text.replace(ref["placeholder"], token)

    text = clean_rendered_sentence(text)
    text = remove_redundant_single_citation_wrappers(text)
    text = clean_rendered_sentence(text)

    if text.count(TARGET_TOKEN) != 1:
        raise RuntimeError(
            "Target-aware rendering produced an invalid TARGET token count."
        )

    # No TEI placeholders may survive into model input.
    if PLACEHOLDER_RE.search(text):
        raise RuntimeError("A TEI citation placeholder survived target rendering.")

    return text


def sentence_resolution_status(refs):
    """
    Strict target resolution for one original citation sentence.

    Accepted only when every TEI bibliographic ref occurrence has exactly one
    usable bibliography target. This keeps TARGET_CITATION atomic and avoids
    inventing a target inside unresolved TEI clusters.
    """
    if not refs:
        return "no_bibliographic_refs"

    missing = [r for r in refs if len(r["target_ids"]) == 0]
    if missing:
        return "tei_target_missing"

    multi = [r for r in refs if len(r["target_ids"]) > 1]
    if multi:
        return "tei_multiple_targets_in_single_ref"

    return "resolved"


def extract_target_aware_sentence_groups_from_xml(xml_path):
    """
    Return:
      1) resolved ORIGINAL sentence groups, each containing one target-aware
         instance per resolvable TEI citation occurrence;
      2) sentence-level target-resolution audit records.
    """
    parser = etree.XMLParser(
        recover=True,
        huge_tree=True,
        resolve_entities=False,
    )

    root = etree.parse(str(xml_path), parser).getroot()

    paragraphs = root.xpath(
        ".//tei:text/tei:body//tei:p",
        namespaces=TEI_NS,
    )

    groups = []
    sentence_audit = []

    for paragraph_index, paragraph in enumerate(paragraphs, start=1):
        marked_paragraph, ref_meta = paragraph_with_target_placeholders(paragraph)
        if not marked_paragraph:
            continue

        sentences = sent_tokenize(marked_paragraph, language="english")

        for sentence_index, marked_sentence in enumerate(sentences, start=1):
            if "__OA_BIBREF_" not in marked_sentence:
                continue

            refs = refs_in_marked_sentence(marked_sentence, ref_meta)
            if refs is None:
                sentence_audit.append({
                    "paragraph_index": paragraph_index,
                    "sentence_index_in_paragraph": sentence_index,
                    "status": "placeholder_metadata_mismatch",
                    "initial_sentence": "",
                    "n_citations_in_sentence": 0,
                    "raw_targets": "",
                })
                continue

            initial_sentence = render_initial_sentence(marked_sentence, refs)
            word_count = len(initial_sentence.split())

            if word_count < MIN_SENTENCE_WORDS:
                sentence_audit.append({
                    "paragraph_index": paragraph_index,
                    "sentence_index_in_paragraph": sentence_index,
                    "status": "too_short",
                    "initial_sentence": initial_sentence,
                    "n_citations_in_sentence": len(refs),
                    "raw_targets": "|".join(r["raw_target"] for r in refs),
                })
                continue

            if word_count > MAX_SENTENCE_WORDS:
                sentence_audit.append({
                    "paragraph_index": paragraph_index,
                    "sentence_index_in_paragraph": sentence_index,
                    "status": "too_many_words",
                    "initial_sentence": initial_sentence,
                    "n_citations_in_sentence": len(refs),
                    "raw_targets": "|".join(r["raw_target"] for r in refs),
                })
                continue

            if len(initial_sentence) > MAX_SENTENCE_CHARS:
                sentence_audit.append({
                    "paragraph_index": paragraph_index,
                    "sentence_index_in_paragraph": sentence_index,
                    "status": "too_many_chars",
                    "initial_sentence": initial_sentence,
                    "n_citations_in_sentence": len(refs),
                    "raw_targets": "|".join(r["raw_target"] for r in refs),
                })
                continue

            status = sentence_resolution_status(refs)
            if status != "resolved":
                sentence_audit.append({
                    "paragraph_index": paragraph_index,
                    "sentence_index_in_paragraph": sentence_index,
                    "status": status,
                    "initial_sentence": initial_sentence,
                    "n_citations_in_sentence": len(refs),
                    "raw_targets": "|".join(r["raw_target"] for r in refs),
                })
                if STRICT_TARGET_AWARE:
                    continue

            # With STRICT_TARGET_AWARE=True only resolved refs reach here.
            if any(len(r["target_ids"]) != 1 for r in refs):
                continue

            target_instances = []
            for target_occurrence_index, ref in enumerate(refs):
                target_sentence = render_target_aware_sentence(
                    marked_sentence,
                    refs,
                    target_occurrence_index,
                )

                target_instances.append({
                    "sentence": target_sentence,
                    "initial_sentence": initial_sentence,
                    "target_bib_id": ref["target_ids"][0],
                    "target_surface": ref["surface"],
                    "target_occurrence_index": target_occurrence_index + 1,
                    "n_citations_in_sentence": int(len(refs)),
                    "paragraph_index": paragraph_index,
                    "sentence_index_in_paragraph": sentence_index,
                })

            groups.append({
                "initial_sentence": initial_sentence,
                "n_citations_in_sentence": int(len(refs)),
                "paragraph_index": paragraph_index,
                "sentence_index_in_paragraph": sentence_index,
                "instances": target_instances,
            })

            sentence_audit.append({
                "paragraph_index": paragraph_index,
                "sentence_index_in_paragraph": sentence_index,
                "status": "resolved",
                "initial_sentence": initial_sentence,
                "n_citations_in_sentence": len(refs),
                "raw_targets": "|".join(r["raw_target"] for r in refs),
            })

    # Within-paper duplicate ORIGINAL context removal. Keep all target-aware
    # sibling instances for the first occurrence of a duplicated context.
    dedup = {}
    for group in groups:
        key = normalized_sentence_key(group["initial_sentence"])
        if key not in dedup:
            dedup[key] = group

    return list(dedup.values()), sentence_audit

In [ ]:
# ============================================================
# 11. PROCESS SELECTED ARTICLES
# ============================================================

candidate_rows = []
article_audit_rows = []
target_audit_rows = []

for _, work in tqdm(
    selected_articles.iterrows(),
    total=len(selected_articles),
    desc="OpenAlex articles",
):
    work_id = work["openalex_work_id"]

    try:
        xml_path, download_status = download_grobid_xml(work)

        sentence_groups, sentence_audit = (
            extract_target_aware_sentence_groups_from_xml(xml_path)
        )

        for audit_row in sentence_audit:
            audit_copy = dict(audit_row)
            audit_copy.update({
                "openalex_work_id": work_id,
                "journal": work["journal"],
                "publication_year": int(work["publication_year"]),
                "doi": work.get("doi"),
                "title": work.get("title"),
            })
            target_audit_rows.append(audit_copy)

        raw_citation_sentence_count = len(sentence_groups)

        # Reproducible cap on ORIGINAL citation sentences, not expanded target
        # instances. Then every target occurrence from a selected sentence is kept.
        if (
            MAX_CITATION_SENTENCES_PER_ARTICLE is not None
            and len(sentence_groups) > MAX_CITATION_SENTENCES_PER_ARTICLE
        ):
            rng = random.Random(
                stable_seed(work_id, "citation_sentences_target_aware")
            )
            selected_indices = sorted(
                rng.sample(
                    range(len(sentence_groups)),
                    MAX_CITATION_SENTENCES_PER_ARTICLE,
                )
            )
            sentence_groups = [sentence_groups[i] for i in selected_indices]

        retained_target_instances = 0

        for group in sentence_groups:
            for instance in group["instances"]:
                retained_target_instances += 1

                candidate_rows.append({
                    "Source_Paper_ID": work_id,
                    "sentence": instance["sentence"],
                    "initial_sentence": instance["initial_sentence"],
                    "is_citation": 1,
                    "polarity": pd.NA,
                    "source": (
                        safe_folder_name(work["journal"])
                        + "_OpenAlex_2023_2025"
                    ),
                    "sampling_status": (
                        "external_real_journal_unlabeled_openalex_tei_target_aware"
                    ),
                    "journal": work["journal"],
                    "doi": work.get("doi"),
                    "title": work.get("title"),
                    "publication_year": int(work["publication_year"]),
                    "publication_date": work.get("publication_date"),
                    "openalex_work_id": work_id,
                    "n_citations_in_sentence": instance["n_citations_in_sentence"],
                    "is_open_access": bool(work.get("is_open_access")),
                    "license": work.get("license"),
                    "oa_status": work.get("oa_status"),
                })

                # Target-specific information is deliberately kept in the audit,
                # not added to the requested annotation CSV schema.
                target_audit_rows.append({
                    "openalex_work_id": work_id,
                    "journal": work["journal"],
                    "publication_year": int(work["publication_year"]),
                    "doi": work.get("doi"),
                    "title": work.get("title"),
                    "paragraph_index": instance["paragraph_index"],
                    "sentence_index_in_paragraph": instance[
                        "sentence_index_in_paragraph"
                    ],
                    "status": "target_instance_retained_before_global_dedup",
                    "initial_sentence": instance["initial_sentence"],
                    "target_aware_sentence": instance["sentence"],
                    "target_bib_id": instance["target_bib_id"],
                    "target_surface": instance["target_surface"],
                    "target_occurrence_index": instance[
                        "target_occurrence_index"
                    ],
                    "n_citations_in_sentence": instance[
                        "n_citations_in_sentence"
                    ],
                    "raw_targets": "",
                })

        article_audit_rows.append({
            "openalex_work_id": work_id,
            "journal": work["journal"],
            "publication_year": int(work["publication_year"]),
            "doi": work.get("doi"),
            "title": work.get("title"),
            "status": "success",
            "download_status": download_status,
            "raw_unique_resolved_citation_sentences": raw_citation_sentence_count,
            "retained_original_citation_sentences": len(sentence_groups),
            "retained_target_aware_instances": retained_target_instances,
            "tei_path": str(xml_path),
            "error": "",
        })

    except Exception as exc:
        article_audit_rows.append({
            "openalex_work_id": work_id,
            "journal": work["journal"],
            "publication_year": int(work["publication_year"]),
            "doi": work.get("doi"),
            "title": work.get("title"),
            "status": "failed",
            "download_status": "",
            "raw_unique_resolved_citation_sentences": 0,
            "retained_original_citation_sentences": 0,
            "retained_target_aware_instances": 0,
            "tei_path": "",
            "error": repr(exc),
        })

article_audit_df = pd.DataFrame(article_audit_rows)
if not article_audit_df.empty:
    article_audit_df.to_csv(
        AUDIT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

if target_audit_rows:
    pd.DataFrame(target_audit_rows).to_csv(
        TARGET_AUDIT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

if not candidate_rows:
    raise RuntimeError(
        "Δεν εξήχθη κανένα strict target-aware citation instance. "
        "Έλεγξε τα audit CSV."
    )

candidates = pd.DataFrame(candidate_rows)

In [ ]:
# ============================================================
# 12. GLOBAL DEDUPLICATION OF TARGET-AWARE INSTANCES
# ============================================================

_assert_target_aware_sentences(candidates["sentence"], "candidate sentences")

candidates["_norm"] = candidates["sentence"].map(normalized_sentence_key)

before_global_dedup = len(candidates)

candidates = (
    candidates
    .drop_duplicates(subset=["_norm"], keep="first")
    .drop(columns=["_norm"])
    .reset_index(drop=True)
)

after_global_dedup = len(candidates)

print("\nGLOBAL TARGET-AWARE DEDUPLICATION")
print("Before:", before_global_dedup)
print("After:", after_global_dedup)
print("Removed:", before_global_dedup - after_global_dedup)

In [ ]:
# ============================================================
# 13. SPLIT BEFORE ANNOTATION AND SAVE BOTH HALVES
# ============================================================

# Exact requested annotation schema: sentence is target-aware, initial_sentence
# is the original extracted context, all other columns are retained as before.
candidates = candidates[ANNOTATION_COLUMNS].copy()

sentiment_half, function_half = _split_before_annotation_50_50(candidates)

# ------------------------------------------------------------
# A) SENTIMENT HALF
# ------------------------------------------------------------
sentiment_annotation_df = sentiment_half[ANNOTATION_COLUMNS].copy()
sentiment_annotation_df["polarity"] = pd.NA
sentiment_annotation_df["sampling_status"] = (
    "external_real_journal_unlabeled_openalex_tei_target_aware"
)

sentiment_annotation_df.to_csv(
    ANNOTATION_CSV,
    index=False,
    encoding="utf-8-sig",
)

# ------------------------------------------------------------
# B) CITATION-FUNCTION HALF
# ------------------------------------------------------------
function_df = _create_function_dataset(function_half)
function_df, function_file_created = _save_function_dataset_once(function_df)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------
print("\nTARGET-AWARE PRE-ANNOTATION 50/50 SPLIT COMPLETE")
print("Total deduplicated target-aware instances:", len(candidates))
print("Absolute half-size difference:", abs(len(sentiment_half) - len(function_half)))

print("\nSENTIMENT HALF — annotate polarity ONLY here")
print(ANNOTATION_CSV)
print("Rows:", len(sentiment_annotation_df))
print("Columns:", list(sentiment_annotation_df.columns))
print("By journal:")
print(sentiment_annotation_df["journal"].value_counts())
print("By year:")
print(sentiment_annotation_df["publication_year"].value_counts().sort_index())

print("\nCITATION-FUNCTION HALF")
print(FINAL_CITATION_FUNCTION_CSV)
print("Rows:", len(function_df))
print("Columns:", list(function_df.columns))
print("follow_up non-empty values:", int(function_df["follow_up"].notna().sum()))

print("\nTARGET-AWARE AUDIT")
print(TARGET_AUDIT_CSV)
if target_audit_rows:
    audit_status_counts = (
        pd.DataFrame(target_audit_rows)["status"].value_counts()
    )
    print(audit_status_counts)

print("\nNEXT STEP:")
print(
    "Συμπλήρωσε ΜΟΝΟ το polarity στο "
    "journal_citation_sentiment_annotation_candidates.csv "
    "με 0 = Negative, 1 = Neutral, 2 = Positive."
)
print(
    "ΜΗΝ αλλάξεις τις sentence / initial_sentence. Μετά βάλε το συμπληρωμένο "
    "annotation CSV στο ίδιο path και ξανατρέξε το script."
)
print(
    "Στο δεύτερο run θα δημιουργηθεί το target-aware "
    "journal_citation_sentiment_real.csv χωρίς νέα OpenAlex requests."
)